# Strategy Lab

Train and evaluate **one** strategy, with every step pinned to the same tickers.

The pinning is the point. `resolve_backtest_universe` draws from whatever is in
the parquet cache, and it deliberately offsets its seed by purpose so training
and backtesting sample *differently*. That is right for a single production run
and wrong for an experiment: train in one cell, backtest in another, and you
have measured two models on two samples without anything telling you so.

A `Lab` resolves the universe once and every method uses it.

## 1. Pin a universe

In [ ]:
%load_ext autoreload
%autoreload 2

from portfolio_agent.lab import Lab

# Three ways to pin, most reproducible first:
#   Lab(tickers=["RELIANCE", "TCS", ...])       exact names
#   Lab(snapshot="universe/experiment.json")    reproduce an earlier run
#   Lab(universe_size=40)                       fresh draw, then frozen
lab = Lab(universe_size=40, name="strategy-lab")
lab

In [ ]:
# Write it down so this experiment can be repeated exactly, on any machine.
lab.save_universe("universe/strategy-lab.json")

print(f"{len(lab.tickers)} tickers, fingerprint {lab.fingerprint}")
print(lab.tickers[:10])

## 2. What can be trained, and with what settings

In [ ]:
print("strategies:", lab.strategies())
print("trainers:  ", lab.trainers())

In [ ]:
# Every hyperparameter the SAC trainer accepts, with its default.
# These are validated on use: an unknown name raises instead of being ignored.
import pandas as pd

pd.Series(Lab.settings("sac"), name="default").to_frame()

## 3. Train

`overrides` are keyword arguments. They are checked against the trainer's own
schema, so `epochs=50` works and `epoch=50` raises with the correct spelling in
the message.

`save=False` keeps the experiment from overwriting whatever `models/` currently
holds — turn it on once a run is worth keeping.

In [ ]:
run = lab.train("india_sac", epochs=30, save=False)

if not run.ok:
    print("FAILED:", run.error)
else:
    print(run.summary())

In [ ]:
# The training curve, epoch by epoch.
history = run.artifact.metrics.get("history", [])
frame = pd.DataFrame(history)
frame.head()

In [ ]:
if not frame.empty:
    axes = frame.plot(
        x="epoch",
        y=[c for c in ("val_sortino", "critic_loss", "actor_loss") if c in frame],
        subplots=True, figsize=(9, 6), legend=True,
    )

## 4. What the run recorded

Provenance travels in the checkpoint, so a model can always be traced back to
the sample it was fitted on — including the universe fingerprint printed above.

In [ ]:
metadata = run.artifact.metadata
{
    "trainer": metadata.get("trainer"),
    "features": metadata.get("feature_names"),
    "has_scaler": metadata.get("feature_scaler") is not None,
    "universe": metadata.get("universe_fingerprint"),
    "best_epoch": metadata.get("best_epoch"),
}

## 5. Backtest on the same names

This is the cell the pinning exists for. `lab.backtest` uses `lab.tickers`, so
the evaluation universe is the training universe rather than a fresh draw.

It needs a saved checkpoint — re-run the training cell with `save=True` first.

In [ ]:
# results = lab.backtest("india_sac")
# pd.Series({k: v for k, v in results.items() if isinstance(v, (int, float))})

## 6. Or both at once

`train_and_backtest` skips the backtest if training failed, rather than
silently reporting numbers from a stale checkpoint.

In [ ]:
# outcome = lab.train_and_backtest("india_sac", epochs=30)
# outcome["run"].summary()